## 5. Preprocessing and Feature Engineering

Preprocessing decisions:

* **Cohort Restriction (Q4 High-Value Segment):** isolate subscribers with `rev_Mean >= USD 70.75` (top 25% of revenue, $N = 24,913$), dropping unique identifier columns and temporary EDA helper columns.

* **Anomaly and Missingness Handling:**
    - Invalid negative equipment ages (eqpdays < 0) are recoded to NaN so they are cleanly imputed rather than distorting splits.
    - Columns with excessive demographic missingness (> 45% missing within Q4, such as numbcars) are dropped to reduce noise.
    - Structured missingness is explicitly preserved via indicator features (has_6m_history for avg6* nulls and missing_count_row for overall demographic sparsity).
    - Missing categorical values are imputed as "Missing", while literal unknown codes ("U", "UNKW") are standardized as "Unknown".

* **Domain-Driven Feature Engineering:**
    - **Bill Shock & Plan Fit:** ovr_rev_ratio (overage share of total bill), mrc_rev_ratio (base plan share of total bill), ovr_mou_ratio (overage share of minutes), and is_high_overage (ovr_rev_ratio > 0.25).
    - **Equipment Lifecycle:** eqp_to_tenure_ratio (ratio of handset age to total customer tenure) and is_old_handset (eqpdays > 365).
    - **Behavioral & Quality Signals:** mou_trend_ratio (relative usage change), rev_trend_ratio (relative revenue change), rev_per_mou (effective cost per minute), and call_failure_rate (dropped + blocked calls over total voice attempts).

* **Split and Pipeline Architecture:** I perform a stratified 80/20 train-test split before fitting any imputer, encoder, or scaler, and retain unscaled business attributes (test_meta) for downstream Revenue-at-Risk evaluation.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


In [2]:
import os
from pathlib import Path

current_dir = Path(os.getcwd())
client_path = current_dir / "telecom" / "Client.csv"
record_path = current_dir / "telecom" / "Record.csv"

for p in (client_path, record_path):
    print(f"{'OK ' if p.exists() else 'MISSING '} {p}")

OK  d:\Estudo\Projetos\telecom-customer-analytics-ml\telecom\Client.csv
OK  d:\Estudo\Projetos\telecom-customer-analytics-ml\telecom\Record.csv


In [3]:
client = pd.read_csv(client_path)
record = pd.read_csv(record_path)

print(f"Client: {client.shape}")
print(f"Record: {record.shape}")

Client: (100000, 50)
Record: (100000, 51)


In [4]:
df = record.merge(client, on='Customer_ID', how='inner')
print(f"Merged: {df.shape}")
df.head()

Merged: (100000, 100)


,rev_Mean,mou_Mean,totmrc_Mean,da_Mean,ovrmou_Mean,ovrrev_Mean,vceovr_Mean,datovr_Mean,roam_Mean,change_mou,...,dwllsize,forgntvl,ethnic,kid0_2,kid3_5,kid6_10,kid11_15,kid16_17,creditcd,eqpdays
0,23.9975,219.25,22.500,0.2475,0.00,0.0,0.0,0.0,0.0,-157.25,...,A,0.0,N,U,U,U,U,U,Y,361.0
1,57.4925,482.75,37.425,0.2475,22.75,9.1,9.1,0.0,0.0,532.25,...,A,0.0,Z,U,U,U,U,U,Y,240.0
2,16.9900,10.25,16.990,0.0000,0.00,0.0,0.0,0.0,0.0,-4.25,...,A,0.0,N,U,Y,U,U,U,Y,1504.0
3,38.0000,7.50,38.000,0.0000,0.00,0.0,0.0,0.0,0.0,-1.50,...,D,0.0,U,Y,U,U,U,U,Y,1812.0
4,55.2300,570.50,71.980,0.0000,0.00,0.0,0.0,0.0,0.0,38.50,...,O,0.0,I,U,U,U,U,U,Y,434.0


### 5.1 Cohort Filtering (Q4 High-Value) and Column Cleanup

In [5]:
q75_rev = df["rev_Mean"].quantile(0.75)
df_q4 = df[df["rev_Mean"] >= q75_rev].copy()

In [6]:
# Remove temporary columns created in EDA.
eda_temp_cols = ["months_bin", "eqp_bin", "rev_quartile", "ovr_rev_ratio"]
df_q4.drop(columns=[c for c in eda_temp_cols if c in df_q4.columns], inplace=True)

In [7]:
id_cols = [
    c for c in df_q4.columns
    if ("id" in c.lower() and df_q4[c].nunique() == len(df_q4))
    or c.lower() in ["customer_id", "cust_id", "id"]
]
print(f"Q4 Cohort shape before cleanup: {df_q4.shape}")
print(f"Dropping ID columns: {id_cols}")
df_q4.drop(columns=id_cols, inplace=True)

Q4 Cohort shape before cleanup: (24913, 100)
Dropping ID columns: ['Customer_ID']


### 5.2 Anomaly Cleaning and Missingness Indicators

In [8]:
neg_eqp = (df_q4["eqpdays"] < 0).sum()
df_q4.loc[df_q4["eqpdays"] < 0, "eqpdays"] = np.nan
print(f"Negative eqpdays recoded to NaN: {neg_eqp}")


Negative eqpdays recoded to NaN: 61


In [9]:
neg_mrc = (df_q4["totmrc_Mean"] < 0).sum()
print(f"Negative totmrc_Mean in Q4: {neg_mrc}")

Negative totmrc_Mean in Q4: 2


In [10]:
df_q4["missing_count_row"] = df_q4.isnull().sum(axis=1)
if "avg6rev" in df_q4.columns:
    df_q4["has_6m_history"] = df_q4["avg6rev"].notna().astype(int)

In [11]:
missing_pct_q4 = df_q4.isnull().mean()
high_missing_cols = missing_pct_q4[missing_pct_q4 > 0.45].index.tolist()
print(f"Dropping columns with >45% missing in Q4: {high_missing_cols}")
df_q4.drop(columns=high_missing_cols, inplace=True)

Dropping columns with >45% missing in Q4: ['numbcars', 'dwllsize']


In [12]:
cat_cols_raw = df_q4.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols_raw:
    df_q4[col] = df_q4[col].replace({"UNKW": "Unknown"})

### 5.3 Business-Driven Feature Engineering (Row-Level)

In [13]:
eps = 1e-5

df_q4["ovr_rev_ratio"] = (df_q4["ovrrev_Mean"] / (df_q4["rev_Mean"] + eps)).clip(0, 1)
df_q4["mrc_rev_ratio"] = (df_q4["totmrc_Mean"] / (df_q4["rev_Mean"] + eps)).clip(0, 1.5)
df_q4["ovr_mou_ratio"] = (df_q4["ovrmou_Mean"] / (df_q4["mou_Mean"] + 1.0)).clip(0, 1)
df_q4["is_high_overage"] = (df_q4["ovr_rev_ratio"] > 0.25).astype(int)

In [14]:
df_q4["eqp_to_tenure_ratio"] = (df_q4["eqpdays"] / (df_q4["months"] * 30.0 + 1.0)).clip(0, 2)
df_q4["is_old_handset"] = (df_q4["eqpdays"] > 365).astype(int)

In [15]:
df_q4["mou_trend_ratio"] = (df_q4["change_mou"] / (df_q4["mou_Mean"] + 1.0)).clip(-5, 5)
df_q4["rev_trend_ratio"] = (df_q4["change_rev"] / (df_q4["rev_Mean"] + 1.0)).clip(-5, 5)
df_q4["rev_per_mou"] = (df_q4["rev_Mean"] / (df_q4["mou_Mean"] + 1.0)).clip(0, 100)

In [16]:
if all(c in df_q4.columns for c in ["drop_vce_Mean", "blck_vce_Mean", "comp_vce_Mean"]):
    total_vce = df_q4["comp_vce_Mean"] + df_q4["drop_vce_Mean"] + df_q4["blck_vce_Mean"]
    df_q4["call_failure_rate"] = ((df_q4["drop_vce_Mean"] + df_q4["blck_vce_Mean"]) / (total_vce + 1.0)).clip(0, 1)

### 5.4 Stratified Train / Test Split (80/20) and Business Metadata

In [17]:
TARGET = "churn"
X = df_q4.drop(columns=[TARGET])
y = df_q4[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [18]:
meta_cols = [
    c for c in [
        "rev_Mean", "totmrc_Mean", "ovrrev_Mean", "ovr_rev_ratio",
        "eqpdays", "asl_flag", "change_mou", "months", "hnd_price"
    ]
    if c in df_q4.columns
]
test_meta = df_q4.loc[X_test.index, meta_cols].copy()
test_meta["churn"] = y_test.values

### 5.5 Preprocessing Pipelines (Tree and Linear)

In [19]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [20]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_features)
    ],
    remainder="drop"
)

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features)
    ],
    remainder="drop"
)

### 5.6 Sanity Check

In [21]:
X_train_tree_check = tree_preprocessor.fit_transform(X_train)
X_test_tree_check = tree_preprocessor.transform(X_test)

In [22]:
print("\nPreprocessing Sanity Check:")
print(f"X_train shape: {X_train.shape} | y_train churn rate: {y_train.mean()*100:.2f}%")
print(f"X_test  shape: {X_test.shape}  | y_test  churn rate: {y_test.mean()*100:.2f}%")
print(f"Numeric features: {len(num_features)} | Categorical features: {len(cat_features)}")
print(f"Remaining NaNs (Train tree-ready): {np.isnan(X_train_tree_check).sum()}")
print(f"Remaining NaNs (Test  tree-ready): {np.isnan(X_test_tree_check).sum()}")
print(f"Test set Monthly Revenue-at-Risk (actual churners): USD {test_meta[test_meta['churn']==1]['rev_Mean'].sum():,.2f}")


Preprocessing Sanity Check:
X_train shape: (19930, 108) | y_train churn rate: 48.61%
X_test  shape: (4983, 108)  | y_test  churn rate: 48.61%
Numeric features: 88 | Categorical features: 20
Remaining NaNs (Train tree-ready): 0
Remaining NaNs (Test  tree-ready): 0
Test set Monthly Revenue-at-Risk (actual churners): USD 281,720.55


In [23]:
engineered_cols = [
    "ovr_rev_ratio", "mrc_rev_ratio", "ovr_mou_ratio", "is_high_overage",
    "eqp_to_tenure_ratio", "is_old_handset", "mou_trend_ratio",
    "rev_trend_ratio", "rev_per_mou", "call_failure_rate",
    "missing_count_row", "has_6m_history"
]
eng_available = [c for c in engineered_cols if c in X_train.columns]
corr_check = X_train[eng_available].corrwith(y_train).sort_values()
print("\nEngineered features correlation with churn (Train set only):")
print(corr_check.round(4))


Engineered features correlation with churn (Train set only):
mrc_rev_ratio         -0.0979
mou_trend_ratio       -0.0401
has_6m_history         0.0138
missing_count_row      0.0155
rev_trend_ratio        0.0290
rev_per_mou            0.0338
call_failure_rate      0.0429
is_high_overage        0.0697
ovr_rev_ratio          0.0874
is_old_handset         0.0897
eqp_to_tenure_ratio    0.0957
ovr_mou_ratio          0.1035
dtype: float64


In [25]:
import os

os.makedirs("telecom/processed", exist_ok=True)

X_train.to_parquet("telecom/processed/X_train.parquet")
X_test.to_parquet("telecom/processed/X_test.parquet")
y_train.to_frame(name="churn").to_parquet("telecom/processed/y_train.parquet")
y_test.to_frame(name="churn").to_parquet("telecom/processed/y_test.parquet")
test_meta.to_parquet("telecom/processed/test_meta.parquet")